# 01 — Data quality and the calendar leak

**Purpose.** Establish what is in the dataset, reconstruct the contact calendar the raw file leaves
implicit, and produce the evidence for this project's central claim: the macro-economic block is a
proxy for the contact month, so a random train/test split measures something a deployed campaign
could never use.

**How to read this notebook.** Every computation calls into `term_deposit`. Nothing analytical is
defined here that the pipeline does not also use — if a number appears below, the same function
produced it in `scripts/train.py`. That is deliberate: a notebook that reimplements the pipeline is
a notebook that can silently disagree with it.

**Prerequisite.** Run `make data` (or `uv run python scripts/prepare_data.py`) once, so the raw CSV
is present and checksum-verified.

**Outline**
1. Load and validate
2. Target and class imbalance
3. Data quality: what `unknown` means here
4. The `pdays` sentinel
5. Reconstructing the contact calendar
6. The leak: macro features are a calendar identifier
7. Base-rate drift across the campaign
8. What this implies for the evaluation protocol

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from term_deposit import constants
from term_deposit.config import load_config
from term_deposit.data.schema import summarise_quality
from term_deposit.features.calendar import macro_period_collinearity, period_summary
from term_deposit.pipelines.experiment import prepare_dataset
from term_deposit.utils.logging import configure_logging

configure_logging("WARNING", force=True)
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)
plt.rcParams["figure.dpi"] = 110

# Notebooks live two directories below the repository root.
ROOT = Path.cwd().parents[1] if Path.cwd().name == "exploratory" else Path.cwd()
config = load_config([ROOT / "configs" / "base.yaml"], project_root=ROOT)
config.paths.raw_csv

## 1. Load and validate

`prepare_dataset` is the same entry point the training pipeline uses. It verifies the file's
SHA-256, enforces the declared table schema, drops `duration`, and attaches the reconstructed
calendar. If any of those fail, it raises here rather than producing a quietly wrong analysis.

In [ ]:
prepared = prepare_dataset(config, download=False)
frame = prepared.frame

print(f"rows          {len(frame):,}")
print(f"columns       {frame.shape[1]}")
print(f"base rate     {prepared.labels.mean():.4f}")
print(f"periods       {prepared.period_key.nunique()} "
      f"({prepared.period_key.min()} .. {prepared.period_key.max()})")
print(f"sha256        {prepared.checksum}")
frame.head()

`duration` is already gone. It records how long the call lasted, so it is only known *after* the
outcome it would be used to predict. Excluding it is the leak that every published analysis of this
dataset catches; the rest of this notebook is about the one that usually goes unnoticed.

In [ ]:
print("dropped before modelling:", config.data.drop_columns)
print("duration present:", "duration" in frame.columns)
print(f"\nmodelling features ({len(config.features.input_columns())}):")
print("  numeric     ", list(config.features.numeric_columns()))
print("  categorical ", list(config.features.categorical_columns()))

## 2. Target and class imbalance

About one contact in nine converts. That number is what every metric in this project is measured
against: a model with no skill achieves an average precision equal to the base rate, and a lift of
exactly 1.0.

In [ ]:
counts = frame[constants.TARGET_COLUMN].value_counts()
shares = frame[constants.TARGET_COLUMN].value_counts(normalize=True)
summary = pd.DataFrame({"n": counts, "share": shares.round(4)})

fig, ax = plt.subplots(figsize=(4.5, 3.2))
ax.bar(summary.index.astype(str), summary["n"], color=["#4C78A8", "#F58518"])
for i, (n, share) in enumerate(zip(summary["n"], summary["share"], strict=True)):
    ax.text(i, n, f"{n:,}\n{share:.1%}", ha="center", va="bottom", fontsize=9)
ax.set(ylabel="contacts", title="Subscription outcome")
ax.margins(y=0.15)
plt.show()

summary

## 3. Data quality: `unknown` is an answer, not a gap

There is not a single null in this file. What looks like missingness is a literal `"unknown"`
category — a customer who declined to answer, or a field the CRM never captured. Those are
different from a missing value, and the project keeps them as their own level rather than imputing
them (see `docs/decisions/0004-unknown-is-a-category-not-a-missing-value.md`).

`default` is the striking one: 20.9% `unknown`, and only three customers in the entire dataset are
recorded as actually being in default.

In [ ]:
quality = summarise_quality(frame)
quality[(quality["n_missing"] > 0) | (quality["n_unknown"] > 0)].sort_values(
    "pct_unknown", ascending=False
).round(2)

In [ ]:
# The `unknown` level is not noise: its subscription rate differs from the observed levels,
# which is the reason for keeping it rather than imputing it away.
rows = []
for column in ("job", "marital", "education", "default", "housing", "loan"):
    grouped = frame.groupby(column, observed=True)[constants.LABEL_COLUMN].agg(["size", "mean"])
    if constants.UNKNOWN_CATEGORY in grouped.index:
        unknown = grouped.loc[constants.UNKNOWN_CATEGORY]
        known = grouped.drop(index=constants.UNKNOWN_CATEGORY)
        rows.append({
            "column": column,
            "n_unknown": int(unknown["size"]),
            "rate_unknown": float(unknown["mean"]),
            "rate_known": float((known["size"] * known["mean"]).sum() / known["size"].sum()),
        })

pd.DataFrame(rows).assign(
    difference=lambda d: d["rate_unknown"] - d["rate_known"]
).round(4)

## 4. The `pdays` sentinel

`pdays` is "days since the client was last contacted in a previous campaign", and `999` codes
"never contacted". That is 96% of rows. Standardised as a raw number, those customers all sit at one
extreme of a variable whose ordering is meaningless for them, and a linear model reads the sentinel
as a distance.

`PdaysSentinelEncoder` splits it into a flag and a real elapsed-days column. The contrast in
subscription rate below is why the flag matters.

In [ ]:
never = frame["pdays"] == constants.PDAYS_NEVER_CONTACTED
print(f"never previously contacted: {never.sum():,} rows ({never.mean():.1%})")
print(f"  subscription rate         {frame.loc[never, constants.LABEL_COLUMN].mean():.4f}")
print(f"previously contacted:       {(~never).sum():,} rows")
print(f"  subscription rate         {frame.loc[~never, constants.LABEL_COLUMN].mean():.4f}")
print(f"  median days since contact {frame.loc[~never, 'pdays'].median():.0f}")

frame.groupby("poutcome", observed=True)[constants.LABEL_COLUMN].agg(
    n="size", subscription_rate="mean"
).round(4)

## 5. Reconstructing the contact calendar

The raw file records `month` but never the year. Its rows are in contact order, so the year can be
recovered exactly: every time the month number decreases relative to the previous row, the calendar
has rolled over.

That single step is what makes the rest of this project possible. Without a calendar key there is no
out-of-time split, no rolling backtest, and no way to ask whether a feature is really a date in
disguise.

The reconstruction is a **label**, used for splitting and reporting only. It is never given to a
model.

In [ ]:
periods = period_summary(frame)
print(f"reconstructed {len(periods)} contact months: "
      f"{periods['contact_period'].min()} .. {periods['contact_period'].max()}")

# Each month appears exactly once, and the sequence is strictly increasing — the signature of a
# genuinely chronological file. A shuffled export would produce hundreds of "years" and the strict
# guard in reconstruct_contact_period would reject it.
ordered = prepared.period_key.to_numpy()
print("row order is non-decreasing in time:", bool((ordered[1:] >= ordered[:-1]).all()))

periods.assign(contact_period=lambda d: d["contact_period"].astype(str)).round(4)

## 6. The leak: the macro block is a calendar identifier

Here is the finding.

`macro_period_collinearity` reports, for each macro feature, the share of its total variance that
lies *between* months. A value of 1.0 means the feature is perfectly constant within any given
month — that it carries no information whatsoever for distinguishing two customers contacted in the
same campaign period.

In [ ]:
collinearity = macro_period_collinearity(frame)
collinearity.round(6)

Four of the five are **exactly** constant within a month. `euribor3m` is a daily rate, so it varies
slightly, but 99.96% of its variance is still between months rather than within them.

The consequence is precise, and it is about deployment rather than statistics: **when you score
today's call list, every customer on it shares today's economy.** Those five columns have zero
variance across the batch. Whatever weight the model puts on them cannot move one customer above
another.

In [ ]:
# Within-month spread of each macro feature, next to its spread across the whole dataset.
rows = []
for feature in constants.MACRO_FEATURES:
    within = frame.groupby(constants.PERIOD_COLUMN, observed=True)[feature].std(ddof=0)
    rows.append({
        "feature": feature,
        "overall_std": float(frame[feature].std(ddof=0)),
        "mean_within_month_std": float(within.mean()),
        "months_with_zero_variance": int((within == 0).sum()),
        "n_months": int(frame[constants.PERIOD_COLUMN].nunique()),
    })
pd.DataFrame(rows).round(4)

## 7. Base-rate drift

The other half of the mechanism. The subscription rate is not stable: it climbs from about 3% at the
start of the campaign to above 50% by 2010, as the bank moved from cold outreach to a warmer,
pre-selected population.

So the dataset contains a strong, genuine relationship — *macro values → this month's base rate* —
that a model can learn and that a random split rewards it for learning.

In [ ]:
plot_data = periods.assign(period=lambda d: d["contact_period"].astype(str))

fig, (top, bottom) = plt.subplots(2, 1, figsize=(11, 7), sharex=True,
                                  gridspec_kw={"height_ratios": [2, 1]})

top.bar(plot_data["period"], plot_data["subscription_rate"], color="#4C78A8", alpha=0.85)
top.set_ylabel("subscription rate")
top.set_title("Subscription rate and macro indicators by contact month")
top.grid(axis="y", alpha=0.3)

twin = top.twinx()
twin.plot(plot_data["period"], plot_data["euribor3m"], color="#E45756", marker="o", linewidth=2,
          label="euribor3m")
twin.plot(plot_data["period"], plot_data["nr.employed"] / 1000, color="#54A24B", marker="s",
          linewidth=2, label="nr.employed / 1000")
twin.set_ylabel("macro indicator")
twin.legend(loc="upper right", fontsize=8)

bottom.bar(plot_data["period"], plot_data["n_contacts"], color="#888888")
bottom.set(ylabel="contacts", xlabel="contact month")
bottom.tick_params(axis="x", rotation=90, labelsize=8)
bottom.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

print(f"lowest  month: {plot_data.loc[plot_data['subscription_rate'].idxmin(), 'period']} "
      f"at {plot_data['subscription_rate'].min():.4f}")
print(f"highest month: {plot_data.loc[plot_data['subscription_rate'].idxmax(), 'period']} "
      f"at {plot_data['subscription_rate'].max():.4f}")

In [ ]:
# The relationship is close to deterministic: knowing the month's macro values is close to
# knowing its base rate.
correlations = {
    feature: float(np.corrcoef(periods[feature], periods["subscription_rate"])[0, 1])
    for feature in constants.MACRO_FEATURES
}
pd.Series(correlations, name="corr(monthly macro value, monthly base rate)").round(4).to_frame()

Note that most of the data is early: the first four months alone hold 58% of all rows, at base rates
between 3% and 6%. The high-conversion months of 2010 are small. Any pooled metric is therefore
dominated by the 2008 population, while the *deployment* question is about the most recent one —
another reason a single pooled number is a poor summary here.

In [ ]:
share = periods.assign(period=lambda d: d["contact_period"].astype(str))[
    ["period", "n_contacts", "share_of_rows", "subscription_rate"]
]
share["cumulative_share"] = share["share_of_rows"].cumsum()
share.head(8).round(4)

## 8. What this implies for the evaluation protocol

Putting the two halves together:

| Fact | Consequence |
|---|---|
| Macro features are constant within a month | They cannot rank customers inside a scored batch |
| Base rate drifts 3% → 57% across months | Those features strongly predict the *pooled* label |
| A random split shares months across train and test | The model is rewarded for learning the calendar |

So a pooled ROC-AUC computed on a random split overstates the part of the score that would survive
deployment. The project's response is not to delete the macro block — that would trade one
unjustified assumption for another — but to **measure the difference**:

- report ranking quality *within* each month alongside the pooled value
  (`term_deposit.evaluation.metrics.within_period_metrics`);
- make the chronological split the default protocol, and keep the random one for comparison;
- select models on a rolling-origin backtest that retrains monthly;
- ship a `client_only` feature set so the macro block's contribution can be measured directly.

Notebook `02-model-evaluation.ipynb` runs those protocols and quantifies the gap.

To reproduce the comparison from the command line:

```bash
uv run python scripts/train.py --set split.strategy=random
uv run python scripts/train.py --set split.strategy=out_of_time
uv run python scripts/train.py --set features.feature_set=client_only
```